In [7]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"  # JDK 17 (pyspark 4.x exige Java 17+)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"  # winutils.exe/hadoop.dll para o Spark funcionar com FS local no Windows
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

import os
os.environ['SPARK_LOCAL_IP'] = '192.168.15.16'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, greatest, lit
from pyspark.sql.functions import col, sum, when
from pyspark.sql.functions import col, when, sum as _sum, avg, max as _max, min as _min, stddev, percentile_approx
from pyspark.sql import Window
from pyspark.sql.functions import lag

spark = SparkSession \
    .builder \
    .appName("HomeCredit_Instalments_Payments") \
    .master("local[1]") \
    .config("spark.driver.host", "192.168.15.16") \
    .config("spark.driver.bindAddress", "192.168.15.16") \
    .getOrCreate()

print(spark.version)

4.2.0


In [8]:
# Cell 1 original (carrega só o instalments_payments.csv):
file_path_instalments_payments = r"C:\Users\muril\OneDrive\Desktop\Fonte\installments_payments.csv"
dados = spark.read.csv(file_path_instalments_payments, header=True, inferSchema=True)

dados.createOrReplaceTempView("dados")
dados.show(5)

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|
+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+
|   1054186|    161674|                   1.0|                    6|        -1180.0|           -1187.0|       6948.36|    6948.36|
|   1330831|    151639|                   0.0|                   34|        -2156.0|           -2156.0|      1716.525|   1716.525|
|   2085231|    193053|                   2.0|                    1|          -63.0|             -63.0|       25425.0|    25425.0|
|   2452527|    199697|                   1.0|                    3|        -2418.0|           -2426.0|      24350.13|   24350.13|
|   2714724|    167756|                   1.0|                    2|        -1383.0

Criando flags alternativas

In [9]:
dados = spark.sql("""

    select
        *,
        DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT as ATRASO_DIAS,
        AMT_PAYMENT / nullif(AMT_INSTALMENT, 0) as RATIO_PAGAMENTO
    from dados

""")

dados.createOrReplaceTempView("dados")
dados.show(5)

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+-----------+------------------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|ATRASO_DIAS|   RATIO_PAGAMENTO|
+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+-----------+------------------+
|   1054186|    161674|                   1.0|                    6|        -1180.0|           -1187.0|       6948.36|    6948.36|       -7.0|               1.0|
|   1330831|    151639|                   0.0|                   34|        -2156.0|           -2156.0|      1716.525|   1716.525|        0.0|               1.0|
|   2085231|    193053|                   2.0|                    1|          -63.0|             -63.0|       25425.0|    25425.0|        0.0|               1.0|
|   2452527|    199697|     

Criando as flags de janelas temporais

In [10]:
dados = spark.sql("""

    select
        *,
        case when DAYS_INSTALMENT >= -90   then 1 else 0 end as flag_ultimos_3_meses,
        case when DAYS_INSTALMENT >= -180  then 1 else 0 end as flag_ultimos_6_meses,
        case when DAYS_INSTALMENT >= -270  then 1 else 0 end as flag_ultimos_9_meses,
        case when DAYS_INSTALMENT >= -360  then 1 else 0 end as flag_ultimos_12_meses,
        case when DAYS_INSTALMENT >= -540  then 1 else 0 end as flag_ultimos_18_meses,
        case when DAYS_INSTALMENT >= -720  then 1 else 0 end as flag_ultimos_24_meses,
        case when DAYS_INSTALMENT >= -1080 then 1 else 0 end as flag_ultimos_36_meses
    from dados

""")

dados.createOrReplaceTempView("dados")

colunas_flags = ['flag_ultimos_3_meses', 'flag_ultimos_6_meses', 'flag_ultimos_9_meses',
                  'flag_ultimos_12_meses', 'flag_ultimos_18_meses', 'flag_ultimos_24_meses',
                  'flag_ultimos_36_meses']

#### Flags categóricas de atraso

In [11]:
dados = spark.sql("""

    select
        *,
        case when ATRASO_DIAS <= 0 then 1 else 0 end as flag_atraso_em_dia,
        case when ATRASO_DIAS between 1 and 15 then 1 else 0 end as flag_atraso_1_15,
        case when ATRASO_DIAS between 16 and 30 then 1 else 0 end as flag_atraso_16_30,
        case when ATRASO_DIAS between 31 and 60 then 1 else 0 end as flag_atraso_31_60,
        case when ATRASO_DIAS between 61 and 90 then 1 else 0 end as flag_atraso_61_90,
        case when ATRASO_DIAS > 90 then 1 else 0 end as flag_atraso_90_mais,
        case when RATIO_PAGAMENTO >= 0.99 then 1 else 0 end as flag_pago_integral,
        case when RATIO_PAGAMENTO < 0.99 then 1 else 0 end as flag_pago_parcial
    from dados

""")

dados.createOrReplaceTempView("dados")
dados.show(5)

+----------+----------+----------------------+---------------------+---------------+------------------+--------------+-----------+-----------+------------------+--------------------+--------------------+--------------------+---------------------+---------------------+---------------------+---------------------+------------------+----------------+-----------------+-----------------+-----------------+-------------------+------------------+-----------------+
|SK_ID_PREV|SK_ID_CURR|NUM_INSTALMENT_VERSION|NUM_INSTALMENT_NUMBER|DAYS_INSTALMENT|DAYS_ENTRY_PAYMENT|AMT_INSTALMENT|AMT_PAYMENT|ATRASO_DIAS|   RATIO_PAGAMENTO|flag_ultimos_3_meses|flag_ultimos_6_meses|flag_ultimos_9_meses|flag_ultimos_12_meses|flag_ultimos_18_meses|flag_ultimos_24_meses|flag_ultimos_36_meses|flag_atraso_em_dia|flag_atraso_1_15|flag_atraso_16_30|flag_atraso_31_60|flag_atraso_61_90|flag_atraso_90_mais|flag_pago_integral|flag_pago_parcial|
+----------+----------+----------------------+---------------------+------------

#### Flag de instabilidade de contrato (NUM_INSTALMENT_VERSION mudando)

In [12]:
janela_prev = Window.partitionBy("SK_ID_PREV").orderBy("NUM_INSTALMENT_NUMBER")

dados = dados.withColumn(
    "flag_mudanca_versao",
    when(col("NUM_INSTALMENT_VERSION") != lag("NUM_INSTALMENT_VERSION").over(janela_prev), 1).otherwise(0)
)

dados.createOrReplaceTempView("dados")

#### Estatísticas gerais (sem estar nas janelas)

In [13]:
colunas_numericas_inst = ['ATRASO_DIAS', 'RATIO_PAGAMENTO', 'AMT_INSTALMENT', 'AMT_PAYMENT']

expressoes_gerais = []

for coluna in colunas_numericas_inst:
    expressoes_gerais.append(avg(col(coluna)).alias(f"MEAN_{coluna}"))
    expressoes_gerais.append(percentile_approx(col(coluna), 0.5).alias(f"MEDIAN_{coluna}"))
    expressoes_gerais.append(_max(col(coluna)).alias(f"MAX_{coluna}"))
    expressoes_gerais.append(_min(col(coluna)).alias(f"MIN_{coluna}"))
    expressoes_gerais.append(stddev(col(coluna)).alias(f"STD_{coluna}"))

# Contagem total de parcelas do cliente — importante pra dar contexto às médias
expressoes_gerais.append(_sum(lit(1)).alias("QTD_TOTAL_PARCELAS"))
expressoes_gerais.append(_sum(col("flag_mudanca_versao")).alias("QTD_MUDANCAS_VERSAO"))

expressoes_gerais = tuple(expressoes_gerais)

book_installments_geral = dados.groupBy("SK_ID_CURR").agg(*expressoes_gerais).orderBy("SK_ID_CURR")

print((book_installments_geral.count(), len(book_installments_geral.columns)))
book_installments_geral.createOrReplaceTempView("df_temp01")
book_installments_geral.show(5)

(339587, 23)
+----------+-------------------+------------------+---------------+---------------+------------------+--------------------+----------------------+-------------------+-------------------+-------------------+-------------------+---------------------+------------------+------------------+------------------+------------------+------------------+---------------+---------------+------------------+------------------+-------------------+
|SK_ID_CURR|   MEAN_ATRASO_DIAS|MEDIAN_ATRASO_DIAS|MAX_ATRASO_DIAS|MIN_ATRASO_DIAS|   STD_ATRASO_DIAS|MEAN_RATIO_PAGAMENTO|MEDIAN_RATIO_PAGAMENTO|MAX_RATIO_PAGAMENTO|MIN_RATIO_PAGAMENTO|STD_RATIO_PAGAMENTO|MEAN_AMT_INSTALMENT|MEDIAN_AMT_INSTALMENT|MAX_AMT_INSTALMENT|MIN_AMT_INSTALMENT|STD_AMT_INSTALMENT|  MEAN_AMT_PAYMENT|MEDIAN_AMT_PAYMENT|MAX_AMT_PAYMENT|MIN_AMT_PAYMENT|   STD_AMT_PAYMENT|QTD_TOTAL_PARCELAS|QTD_MUDANCAS_VERSAO|
+----------+-------------------+------------------+---------------+---------------+------------------+-----------------

#### Estatísticas por janelas temporais

In [14]:
expressoes_temporal = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_numericas_inst:
        base = when(col(flag) == 1, col(coluna))

        expressoes_temporal.append(avg(base).alias(f"MEAN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(percentile_approx(base, 0.5).alias(f"MEDIAN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(_max(base).alias(f"MAX_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(_min(base).alias(f"MIN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(stddev(base).alias(f"STD_{coluna}_{sufixo_janela}"))

expressoes_temporal = tuple(expressoes_temporal)

book_installments_temporal = dados.groupBy("SK_ID_CURR").agg(*expressoes_temporal).orderBy("SK_ID_CURR")

print((book_installments_temporal.count(), len(book_installments_temporal.columns)))
book_installments_temporal.createOrReplaceTempView("df_temp02")
book_installments_temporal.show(5)

(339587, 141)
+----------+-------------------+---------------------+------------------+------------------+------------------+-----------------------+-------------------------+----------------------+----------------------+----------------------+----------------------+------------------------+---------------------+---------------------+---------------------+-------------------+---------------------+------------------+------------------+------------------+-------------------+---------------------+------------------+------------------+------------------+-----------------------+-------------------------+----------------------+----------------------+----------------------+----------------------+------------------------+---------------------+---------------------+---------------------+-------------------+---------------------+------------------+------------------+------------------+-------------------+---------------------+------------------+------------------+------------------+-------------

#### Razões entre as janelas temporais

In [15]:
janelas_ordem = ['U3', 'U6', 'U9', 'U12', 'U18', 'U24', 'U36']
pares_u3_vs_demais = [(janelas_ordem[0], j) for j in janelas_ordem[1:]]
pares_consecutivos = [(janelas_ordem[i], janelas_ordem[i + 1]) for i in range(1, len(janelas_ordem) - 1)]

def gerar_expressoes_razao(df, pares, colunas, metrica='MEAN'):
    colunas_existentes = set(df.columns)
    expressoes = []
    for numerador, denominador in pares:
        for coluna in colunas:
            col_num = f"{metrica}_{coluna}_{numerador}"
            col_den = f"{metrica}_{coluna}_{denominador}"
            if col_num in colunas_existentes and col_den in colunas_existentes:
                expressoes.append(
                    (col(col_num) / when(col(col_den) != 0, col(col_den)))
                    .alias(f"RATIO_{coluna}_{metrica}_{numerador}_{denominador}")
                )
    return expressoes

# ATRASO_DIAS e RATIO_PAGAMENTO são "nível de comportamento" -> MEAN faz mais sentido que SUM
expressoes_razao_a = gerar_expressoes_razao(book_installments_temporal, pares_u3_vs_demais, colunas_numericas_inst, metrica='MEAN')
expressoes_razao_b = gerar_expressoes_razao(book_installments_temporal, pares_consecutivos, colunas_numericas_inst, metrica='MEAN')

book_installments_razoes = book_installments_temporal.select("SK_ID_CURR", *expressoes_razao_a, *expressoes_razao_b)

print((book_installments_razoes.count(), len(book_installments_razoes.columns)))
book_installments_razoes.createOrReplaceTempView("df_temp03")
book_installments_razoes.show(5)

(339587, 45)
+----------+----------------------------+--------------------------------+-------------------------------+----------------------------+----------------------------+--------------------------------+-------------------------------+----------------------------+-----------------------------+---------------------------------+--------------------------------+-----------------------------+-----------------------------+---------------------------------+--------------------------------+-----------------------------+-----------------------------+---------------------------------+--------------------------------+-----------------------------+-----------------------------+---------------------------------+--------------------------------+-----------------------------+----------------------------+--------------------------------+-------------------------------+----------------------------+-----------------------------+---------------------------------+--------------------------------+-

#### Contagens das faixas de atraso por janela temporal

In [16]:
colunas_atraso_flags = ['flag_atraso_em_dia', 'flag_atraso_1_15', 'flag_atraso_16_30',
                          'flag_atraso_31_60', 'flag_atraso_61_90', 'flag_atraso_90_mais']

expressoes_atraso_janela = []

for flag in colunas_flags:
    for coluna in colunas_atraso_flags:
        expressoes_atraso_janela.append(
            _sum(when(col(flag) == 1, col(coluna)).otherwise(0)).alias(f"QTD_{coluna.upper()}_{flag.upper()}")
        )

expressoes_atraso_janela = tuple(expressoes_atraso_janela)

book_installments_atraso_janela = dados.groupBy("SK_ID_CURR").agg(*expressoes_atraso_janela).orderBy("SK_ID_CURR")

print((book_installments_atraso_janela.count(), len(book_installments_atraso_janela.columns)))
book_installments_atraso_janela.createOrReplaceTempView("df_temp04")
book_installments_atraso_janela.show(5)

(339587, 43)
+----------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+--------------------------------------------+------------------------------------------+-------------------------------------------+-------------------------------------------+-------------------

#### Taxa de parcelas pagas integralmente vs parciais

In [17]:
book_installments_taxa_pagamento = dados.groupBy("SK_ID_CURR").agg(
    _sum("flag_pago_integral").alias("QTD_PARCELAS_INTEGRAIS"),
    _sum("flag_pago_parcial").alias("QTD_PARCELAS_PARCIAIS"),
    _sum(lit(1)).alias("QTD_TOTAL_PARCELAS_TAXA")
).withColumn(
    "TAXA_PARCELAS_INTEGRAIS",
    col("QTD_PARCELAS_INTEGRAIS") / when(col("QTD_TOTAL_PARCELAS_TAXA") != 0, col("QTD_TOTAL_PARCELAS_TAXA"))
).withColumn(
    "TAXA_PARCELAS_PARCIAIS",
    col("QTD_PARCELAS_PARCIAIS") / when(col("QTD_TOTAL_PARCELAS_TAXA") != 0, col("QTD_TOTAL_PARCELAS_TAXA"))
)

print((book_installments_taxa_pagamento.count(), len(book_installments_taxa_pagamento.columns)))
book_installments_taxa_pagamento.createOrReplaceTempView("df_temp05")
book_installments_taxa_pagamento.show(5)

(339587, 6)
+----------+----------------------+---------------------+-----------------------+-----------------------+----------------------+
|SK_ID_CURR|QTD_PARCELAS_INTEGRAIS|QTD_PARCELAS_PARCIAIS|QTD_TOTAL_PARCELAS_TAXA|TAXA_PARCELAS_INTEGRAIS|TAXA_PARCELAS_PARCIAIS|
+----------+----------------------+---------------------+-----------------------+-----------------------+----------------------+
|    139128|                    98|                    0|                     98|                    1.0|                   0.0|
|    145504|                    26|                    6|                     32|                 0.8125|                0.1875|
|    197588|                   119|                    0|                    119|                    1.0|                   0.0|
|    173898|                     6|                   12|                     18|     0.3333333333333333|    0.6666666666666666|
|    154034|                   150|                    3|                    154|    

#### Range e CV por janela (dispersão)

In [18]:
expressoes_range_cv = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_numericas_inst:
        base = when(col(flag) == 1, col(coluna))

        expressoes_range_cv.append((_max(base) - _min(base)).alias(f"RANGE_{coluna}_{sufixo_janela}"))
        expressoes_range_cv.append((stddev(base) / when(avg(base) != 0, avg(base))).alias(f"CV_{coluna}_{sufixo_janela}"))

expressoes_range_cv = tuple(expressoes_range_cv)

book_installments_range_cv = dados.groupBy("SK_ID_CURR").agg(*expressoes_range_cv).orderBy("SK_ID_CURR")

print((book_installments_range_cv.count(), len(book_installments_range_cv.columns)))
book_installments_range_cv.createOrReplaceTempView("df_temp06")
book_installments_range_cv.show(5)

(339587, 57)
+----------+--------------------+-------------------+------------------------+---------------------+-----------------------+--------------------+--------------------+------------------+--------------------+-------------------+------------------------+---------------------+-----------------------+--------------------+--------------------+------------------+--------------------+--------------------+------------------------+---------------------+-----------------------+--------------------+--------------------+------------------+---------------------+--------------------+-------------------------+----------------------+------------------------+---------------------+---------------------+------------------+---------------------+--------------------+-------------------------+----------------------+------------------------+---------------------+---------------------+------------------+---------------------+-------------------+-------------------------+----------------------+----

## Join Final

In [19]:
book_installments = spark.sql("""

    select
        d1.SK_ID_CURR,
        d1.*EXCEPT (SK_ID_CURR),
        d2.*EXCEPT (SK_ID_CURR),
        d3.*EXCEPT (SK_ID_CURR),
        d4.*EXCEPT (SK_ID_CURR),
        d5.*EXCEPT (SK_ID_CURR),
        d6.*EXCEPT (SK_ID_CURR)
    from df_temp01 as d1
    left join df_temp02 as d2 on d1.SK_ID_CURR = d2.SK_ID_CURR
    left join df_temp03 as d3 on d1.SK_ID_CURR = d3.SK_ID_CURR
    left join df_temp04 as d4 on d1.SK_ID_CURR = d4.SK_ID_CURR
    left join df_temp05 as d5 on d1.SK_ID_CURR = d5.SK_ID_CURR
    left join df_temp06 as d6 on d1.SK_ID_CURR = d6.SK_ID_CURR

""")

print((book_installments.count(), len(book_installments.columns)))

# Checagem de duplicadas — sempre rodar depois de mexer no join
colunas = book_installments.columns
duplicadas = [c for c in colunas if colunas.count(c) > 1]
print(set(duplicadas))

(339587, 310)
set()


In [20]:
df_temp_installments = book_installments.repartition(1)
df_temp_installments.write.mode("overwrite").parquet("installments_payments_agg.parquet")